In [19]:
import pandas as pd
import numpy as np
import ast

from sklearn.model_selection import GroupShuffleSplit

PATH_CLIPS                          = "../../Data/Videos/Clips/"
PATH_ACTIONS_FILTERED               = "../../Data/Processed/actions_filtered.csv"

PATH_KEYPOINTS                      = "../../Data/Unprocessed/keypoints.csv"
PATH_METRICS                        = "../../Data/Unprocessed/metrics.csv"
PATH_ROI                            = "../../Data/Unprocessed/roi.csv"

PATH_CLS_TRAIN                       = "../../Data/Processed/cls_train.csv"
PATH_CLS_TEST                        = "../../Data/Processed/cls_test.csv"

PATH_ROI_TRAIN                       = "../../Data/Processed/roi_train.csv"
PATH_ROI_TEST                        = "../../Data/Processed/roi_test.csv"

WINDOW_SIZE                         = 4

In [20]:
df_metrics = pd.read_csv(PATH_METRICS)

total_expected_frames = df_metrics["expected"].sum()
total_actual_frames = df_metrics["actual"].sum()
total_coverage = total_actual_frames / total_expected_frames

print(df_metrics[df_metrics["expected"] != df_metrics["actual"]])
print("")

print("Expected frames: ", total_expected_frames)
print("Actual frames:   ", total_actual_frames)
print(f"Coverage:         {total_coverage*100:.2f}%")

    fencer  action_id  start_frame  end_frame  expected  actual   coverage  \
275  RIGHT        997           31         40        10       9  90.000000   
619  RIGHT        768           27         32         6       5  83.333333   

               file  
275  6/20_Right.mp4  
619   5/15_Left.mp4  

Expected frames:  16828
Actual frames:    16826
Coverage:         99.99%


In [21]:
df_keypoints = pd.read_csv(PATH_KEYPOINTS)

df_keypoints["keypoints"] = df_keypoints["keypoints"].apply(
    lambda x: [tuple(p) for p in ast.literal_eval(x)] if isinstance(x, str) else x
)

In [22]:
df_filtered = pd.read_csv(PATH_ACTIONS_FILTERED)

df_merged = df_keypoints.merge(df_filtered, on=["file", "fencer"], how="left")
df_merged = df_merged[
    (df_merged["frame"] >= df_merged["start_frame"]) &
    (df_merged["frame"] <= df_merged["end_frame"])
]

df_merged = df_merged[["file", "fencer", "action_id", "action", "frame", "start_frame", "end_frame", "confidence", "keypoints"]].reset_index(drop=True)
print(df_merged)

                file fencer  action_id        action  frame  start_frame  \
0      1/10_Left.mp4   LEFT          0     NO_ACTION      0            0   
1      1/10_Left.mp4   LEFT          0     NO_ACTION      1            0   
2      1/10_Left.mp4   LEFT          0     NO_ACTION      2            0   
3      1/10_Left.mp4   LEFT          0     NO_ACTION      3            0   
4      1/10_Left.mp4   LEFT          0     NO_ACTION      4            0   
...              ...    ...        ...           ...    ...          ...   
16823   6/9_Left.mp4  RIGHT       1126  SHORT_ATTACK     40           39   
16824   6/9_Left.mp4  RIGHT       1126  SHORT_ATTACK     41           39   
16825   6/9_Left.mp4  RIGHT       1126  SHORT_ATTACK     42           39   
16826   6/9_Left.mp4  RIGHT       1126  SHORT_ATTACK     43           39   
16827   6/9_Left.mp4  RIGHT       1126  SHORT_ATTACK     44           39   

       end_frame  confidence  \
0             22    0.898973   
1             22    0.8

In [23]:
gss = GroupShuffleSplit(
    n_splits=1,
    train_size=0.8,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df_merged, groups=df_merged['file'])
)

df_train = df_merged.iloc[train_idx]
df_test  = df_merged.iloc[test_idx]

In [24]:
def create_sliding_windows(df,window_size=4,stride=1):
    window_rows = []
    window_counter = 0

    # Process each video separately
    for file_name, group in df.groupby(["file", "fencer"]):
        group = group.sort_values(["action_id", "frame"]).reset_index(drop=True)

        n_frames = len(group)
        if n_frames < window_size:
            continue

        for start in range(0, n_frames - window_size + 1, stride):

            end = start + window_size
            window_slice = group.iloc[start:end].copy()

            window_counter += 1
            window_slice["window_id"] = window_counter
            window_slice["action"] = window_slice.iloc[window_size - 1]["action"]

            window_rows.append(window_slice)

    if not window_rows:
        return pd.DataFrame()

    return pd.concat(window_rows, ignore_index=True)

In [25]:
df_train_windowed = create_sliding_windows(df_train, window_size=WINDOW_SIZE)
df_test_windowed  = create_sliding_windows(df_test, window_size=WINDOW_SIZE)

train_counts = df_train_windowed.groupby("action")["window_id"].nunique().sort_values(ascending=False)
test_counts  = df_test_windowed.groupby("action")["window_id"].nunique().sort_values(ascending=False)

print(train_counts)
print("")
print(test_counts)

print("")
print("Number of training action snippets: ", df_train_windowed["window_id"].nunique())
print("Number of testing action snippets:  ", df_test_windowed["window_id"].nunique())

action
NO_ACTION       8149
SHORT_ATTACK    1478
LONG_ATTACK     1330
DIST_PULL        916
PARRY            163
Name: window_id, dtype: int64

action
NO_ACTION       2525
SHORT_ATTACK     629
LONG_ATTACK      400
DIST_PULL        290
PARRY             54
Name: window_id, dtype: int64

Number of training action snippets:  12036
Number of testing action snippets:   3898


In [26]:
def explode_keypoints(df):
    # Expand each tuple into separate x,y columns
    exploded = df["keypoints"].apply(
        lambda kp: [coord for point in kp for coord in point]
    )

    # Create column names: x0, y0, x1, y1, ...
    num_points = len(df.iloc[0]["keypoints"])
    cols = [f"x{i}" for i in range(num_points)] + [f"y{i}" for i in range(num_points)]
    
    new_df = pd.DataFrame(exploded.tolist(), columns=cols)
    return pd.concat([df.drop(columns=["keypoints"]), new_df], axis=1)

In [27]:
df_train_data = df_train_windowed[["file", "fencer", "window_id", "frame", "action", "keypoints"]].copy()
df_test_data = df_test_windowed[["file", "fencer", "window_id", "frame", "action", "keypoints"]].copy()

df_train_data.sort_values(["window_id", "frame"], inplace=True)
df_test_data.sort_values(["window_id", "frame"], inplace=True)

df_train_data = explode_keypoints(df_train_data)
df_test_data = explode_keypoints(df_test_data)

df_train_data.to_csv(PATH_CLS_TRAIN, index=False)
df_test_data.to_csv(PATH_CLS_TEST, index=False)

In [28]:
df_roi = pd.read_csv(PATH_ROI)

gss_roi = GroupShuffleSplit(
    n_splits=1,
    train_size=0.8,
    random_state=42
)

train_idx, test_idx = next(
    gss_roi.split(df_roi, groups=df_roi['file'])
)

df_train_roi = df_roi.iloc[train_idx]
df_test_roi  = df_roi.iloc[test_idx]

df_train_roi.to_csv(PATH_ROI_TRAIN, index=False)
df_test_roi.to_csv(PATH_ROI_TEST, index=False)